# AI Book Recommendation System — Model Training Notebook

This notebook walks through the full machine learning pipeline used by the
Streamlit application, built on the real-world **Kaggle Book Recommendation
Dataset** (arashnic/book-recommendation-dataset):

1. Load & explore the raw dataset (Books.csv, Ratings.csv, Users.csv)
2. Clean and preprocess the data
3. Exploratory Data Analysis (EDA)
4. Train the **Content-Based Filtering** model (TF-IDF + sparse Nearest Neighbors)
5. Train the **Collaborative Filtering** model (Surprise SVD)
6. Evaluate and persist both models

> Before running this notebook, download the dataset from
> https://www.kaggle.com/datasets/arashnic/book-recommendation-dataset and
> place `Books.csv`, `Ratings.csv`, and `Users.csv` inside `../dataset/`.


In [ ]:
# Setup: allow imports from the project root
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)


## 1. Load Raw Data

In [ ]:
from utils.preprocess import load_raw_data

books_raw, ratings_raw, users_raw = load_raw_data(os.path.join(PROJECT_ROOT, "dataset"))

print("Books  :", books_raw.shape)
print("Ratings:", ratings_raw.shape)
print("Users  :", users_raw.shape)
books_raw.head()


In [ ]:
ratings_raw.head()


In [ ]:
users_raw.head()


## 2. Data Quality Check

Before cleaning, let's inspect missing values, duplicates, and invalid values —
the raw Book-Crossing export is known to have missing authors/publishers,
duplicate ISBNs, implicit (0) ratings, and out-of-range publication years
(e.g. 0 or 2038).


In [ ]:
print("Missing values in books:")
print(books_raw[["ISBN", "Book-Title", "Book-Author", "Publisher", "Year-Of-Publication"]].isnull().sum())

print("\nDuplicate ISBNs:", books_raw["ISBN"].duplicated().sum())


In [ ]:
print("Book-Rating value range:", ratings_raw["Book-Rating"].min(), "-", ratings_raw["Book-Rating"].max())
print("Implicit (0) ratings:", (ratings_raw["Book-Rating"] == 0).sum())

years = pd.to_numeric(books_raw["Year-Of-Publication"], errors="coerce")
print("\nInvalid publication years (0 or > current year):")
print(((years <= 0) | (years > 2026)).sum())


## 3. Clean & Preprocess Data

In [ ]:
from utils.preprocess import load_and_clean_all

books, ratings, users = load_and_clean_all(os.path.join(PROJECT_ROOT, "dataset"))

print("Clean Books  :", books.shape)
print("Clean Ratings:", ratings.shape)
print("Clean Users  :", users.shape)
books.head()


## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

books["publisher"].value_counts().head(15).plot(kind="bar", ax=axes[0], color="#6C63FF")
axes[0].set_title("Top 15 Publishers by Book Count")
axes[0].set_ylabel("Count")

ratings["rating"].plot(kind="hist", bins=10, ax=axes[1], color="#FF6584")
axes[1].set_title("Distribution of Explicit Ratings (1-10)")
axes[1].set_xlabel("Rating")

plt.tight_layout()
plt.show()


In [ ]:
print("Average rating overall:", round(ratings["rating"].mean(), 2))
print("Ratings per user (avg):", round(ratings.groupby("user_id").size().mean(), 2))
print("Ratings per book (avg):", round(ratings.groupby("isbn").size().mean(), 2))


## 5. Content-Based Filtering — TF-IDF + Sparse Nearest Neighbors

The real dataset has no genre or description column, so we combine `title`,
`author`, and `publisher` into a single text field, vectorize it with TF-IDF,
and query similarity with a sparse Nearest-Neighbors index.

With ~271K books, a dense book-vs-book similarity matrix would need
271,000² floats (hundreds of GB of RAM), so we deliberately never build one —
`NearestNeighbors` with `metric="cosine"` searches the sparse TF-IDF matrix
directly and only returns the top-N neighbors for a single query book.


In [ ]:
from utils.content_based import train_content_model, save_content_model, build_nn_index, get_content_recommendations

vectorizer, tfidf_matrix, isbn_index = train_content_model(books)
nn_model = build_nn_index(tfidf_matrix)

print("TF-IDF vocabulary size:", len(vectorizer.vocabulary_))
print("TF-IDF matrix shape:", tfidf_matrix.shape, "(sparse)")


In [ ]:
sample_title = books.iloc[0]["title"]
recommendations = get_content_recommendations(sample_title, books, nn_model, tfidf_matrix, isbn_index, top_n=10)

print(f"Top 10 books similar to: {sample_title}\n")
recommendations[["title", "author", "publisher", "avg_rating", "similarity_score"]]


In [ ]:
# Persist the content-based model
save_content_model(vectorizer, tfidf_matrix, isbn_index, models_dir=os.path.join(PROJECT_ROOT, "models"))
print("Saved tfidf.pkl, content_matrix.npz, and content_index.pkl")


## 6. Collaborative Filtering — Surprise SVD

We train a Singular Value Decomposition (SVD) model on the user-book
rating matrix (ISBN as the item key) to learn latent factors and predict
ratings for unseen books. Ratings are explicit values on Book-Crossing's
native 1-10 scale (implicit 0-ratings were dropped during cleaning).


In [ ]:
from utils.collaborative import train_svd_model, save_svd_model, get_personalized_recommendations

svd_model, rmse = train_svd_model(ratings, n_factors=50, n_epochs=20)
print(f"Test RMSE: {rmse:.4f}")


In [ ]:
sample_user_id = int(ratings["user_id"].iloc[0])
personalized = get_personalized_recommendations(sample_user_id, svd_model, books, ratings, top_n=10)

print(f"Top 10 personalized recommendations for user {sample_user_id}\n")
personalized[["title", "author", "publisher", "avg_rating", "predicted_rating"]]


In [ ]:
# Persist the SVD model
save_svd_model(svd_model, models_dir=os.path.join(PROJECT_ROOT, "models"))
print("Saved svd_model.pkl")


## 7. Model Evaluation Summary

- **Content-Based Filtering**: evaluated qualitatively via nearest-neighbor
  similarity — books with overlapping title/author/publisher terms score highest.
  Recommendations for a personalized candidate pool of unrated books are
  scored on demand rather than precomputed for every pair, which is what
  keeps this feasible at 271K books.
- **Collaborative Filtering (SVD)**: evaluated quantitatively via RMSE on a
  held-out test split of the rating matrix (reported above), then refit on
  the full dataset for production use.


In [ ]:
print("=" * 50)
print("MODEL TRAINING SUMMARY")
print("=" * 50)
print(f"Books processed        : {len(books):,}")
print(f"Ratings processed      : {len(ratings):,}")
print(f"Users processed        : {len(users):,}")
print(f"TF-IDF vocabulary size : {len(vectorizer.vocabulary_):,}")
print(f"SVD Test RMSE          : {rmse:.4f}")


## Next Steps

Run the Streamlit application to interact with both models:

```bash
streamlit run app.py
```
